In [ ]:
# CELL 1 — FIXES — Run this first, wait for it to complete
import subprocess
subprocess.run(['pip', 'install', 'imagecorruptions', 'scipy', '-q'], check=True)

import numpy as np
import scipy, scipy.ndimage, sys

# FIX 1
if not hasattr(np, 'float_'):
    np.float_ = np.float64
    print('[FIX 1 APPLIED] np.float_ = np.float64')
else:
    print('[FIX 1 OK] np.float_ already exists')

# FIX 2
from skimage import filters as _skfilters
_orig_gaussian = _skfilters.gaussian
def _patched_gaussian(image, sigma=1, multichannel=None, channel_axis=None, **kwargs):
    if multichannel is not None and channel_axis is None:
        channel_axis = -1 if multichannel else None
    return _orig_gaussian(image, sigma=sigma, channel_axis=channel_axis, **kwargs)
_skfilters.gaussian = _patched_gaussian
import skimage.filters
skimage.filters.gaussian = _patched_gaussian
print('[FIX 2 APPLIED] skimage.filters.gaussian patched')

# FIX 3
if not hasattr(scipy.ndimage, 'filters'):
    scipy.ndimage.filters = scipy.ndimage
if 'scipy.ndimage.filters' not in sys.modules:
    sys.modules['scipy.ndimage.filters'] = scipy.ndimage
print('[FIX 3 APPLIED] scipy.ndimage.filters patched')

print()
print('ALL FIXES DONE. Now run Cell 2.')

In [ ]:
# CELL 2 — FULL EXPERIMENT: glass_blur | BMIA-Adaptive | lr=0.05
import os, csv, time, warnings
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from imagecorruptions import corrupt

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', device)
print('CUDA   :', torch.cuda.is_available())

# ── CONFIG (identical to run / run) ──────────────────────────────────────
SEVERITY              = 5
BATCH_SIZE            = 64
SEED                  = 42
NUM_WORKERS           = 2
K                     = 1000
IMAGES_PER_CORRUPTION = 5000
LR                    = 0.05        # high-lr — the missing row
LAMBDA_INIT           = 0.5
LAMBDA_MIN            = 0.01
LAMBDA_MAX            = 10.0
TAU                   = 0.10
COLLAPSE_DOM_THRESH    = 2.0 / K
COLLAPSE_ACTIVE_THRESH = K // 2
COLLAPSE_HARD_THRESH   = K // 5

np.random.seed(SEED)
torch.manual_seed(SEED)
print('Config OK')

# ── DATASET PATH — auto-discover ──────────────────────────────────────────
VAL_DIR    = '/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val'
LABEL_FILE = '/kaggle/input/imagenet-object-localization-challenge/LOC_val_solution.csv'

if not os.path.isdir(VAL_DIR):
    print('Searching for val dir...')
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'val' in dirs:
            candidate = os.path.join(root, 'val')
            try:
                if any(f.endswith('.JPEG') for f in os.listdir(candidate)):
                    VAL_DIR = candidate
                    print('Found:', VAL_DIR)
                    break
            except:
                continue

if not os.path.exists(LABEL_FILE):
    print('Searching for label file...')
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'LOC_val_solution.csv' in files:
            LABEL_FILE = os.path.join(root, 'LOC_val_solution.csv')
            print('Found:', LABEL_FILE)
            break

assert os.path.isdir(VAL_DIR),    f'VAL_DIR not found: {VAL_DIR}'
assert os.path.exists(LABEL_FILE), f'LABEL_FILE not found: {LABEL_FILE}'
print('VAL_DIR   :', VAL_DIR)
print('LABEL_FILE:', LABEL_FILE)

# ── BUILD SUBSET (same seed = same 5000 images as run/run) ───────────────
synsets, rows = set(), {}
with open(LABEL_FILE, 'r') as f:
    for row in csv.DictReader(f):
        img_id = row['ImageId']
        synset = row['PredictionString'].split()[0]
        synsets.add(synset)
        rows[img_id] = synset

synset_to_idx = {s: i for i, s in enumerate(sorted(synsets))}
img_list = [
    (os.path.join(VAL_DIR, img_id + '.JPEG'), synset_to_idx[syn])
    for img_id, syn in sorted(rows.items())
    if os.path.exists(os.path.join(VAL_DIR, img_id + '.JPEG'))
]

np.random.seed(SEED)
indices    = np.random.choice(len(img_list), IMAGES_PER_CORRUPTION, replace=False)
img_subset = [img_list[i] for i in sorted(indices)]
print(f'Total val images : {len(img_list)}')
print(f'Classes          : {len(synset_to_idx)}')
print(f'Subset size      : {len(img_subset)}')

# ── TRANSFORMS ───────────────────────────────────────────────────────────
NORMALIZE   = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
TO_TENSOR   = transforms.Compose([transforms.ToTensor(), NORMALIZE])
RESIZE_CROP = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])

class CorruptedImageNet(Dataset):
    def __init__(self, img_list, corruption_name, severity):
        self.img_list        = img_list
        self.corruption_name = corruption_name
        self.severity        = severity
    def __len__(self):
        return len(self.img_list)
    def __getitem__(self, idx):
        path, label = self.img_list[idx]
        img    = Image.open(path).convert('RGB')
        img    = RESIZE_CROP(img)
        img_np = np.array(img, dtype=np.uint8)
        img_np = corrupt(img_np, corruption_name=self.corruption_name, severity=self.severity)
        return TO_TENSOR(Image.fromarray(img_np.astype(np.uint8))), label

# ── MODEL UTILS ──────────────────────────────────────────────────────────
def load_resnet50():
    try:
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    except Exception:
        m = models.resnet50(pretrained=True)
    return m.to(device)

def configure_tta(model):
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.train()
            m.requires_grad_(True)
            m.track_running_stats = False
            m.running_mean = None
            m.running_var  = None
    return model

def save_theta0(model):
    t0 = {}
    for nm, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d):
            t0[nm + '.w'] = m.weight.data.clone()
            t0[nm + '.b'] = m.bias.data.clone()
    return t0

def get_bn_params(model):
    return [p for m in model.modules()
            if isinstance(m, nn.BatchNorm2d)
            for p in m.parameters() if p.requires_grad]

def mi_loss(outputs):
    p   = torch.softmax(outputs, dim=1).clamp(min=1e-8)
    hyx = -(p * p.log()).sum(dim=1).mean()
    q   = p.mean(dim=0).clamp(min=1e-8)
    hy  = -(q * q.log()).sum()
    return hyx - hy

def anchor_loss(model, theta0):
    loss = torch.tensor(0.0, device=device)
    for nm, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d):
            loss = loss + (m.weight - theta0[nm + '.w']).pow(2).sum()
            loss = loss + (m.bias   - theta0[nm + '.b']).pow(2).sum()
    return loss

# ── RUN ──────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('RUNNING: glass_blur | BMIA-Adaptive | lr=0.05')
print('NOTE: glass_blur corruption runs on CPU (~40 min). Please wait.')
print('=' * 60)

loader = DataLoader(
    CorruptedImageNet(img_subset, 'glass_blur', SEVERITY),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

model  = load_resnet50()
configure_tta(model)
theta0 = save_theta0(model)
opt    = torch.optim.SGD(get_bn_params(model), lr=LR, momentum=0.9)

lam       = LAMBDA_INIT
all_preds  = []
all_labels = []
hyx_sum    = 0.0
p_sum      = torch.zeros(K)
n_total    = 0
t_start    = time.time()

for batch_idx, (images, labels) in enumerate(loader):
    images = images.to(device)

    opt.zero_grad()
    outs = model(images)
    (mi_loss(outs) + lam * anchor_loss(model, theta0)).backward()
    opt.step()

    with torch.no_grad():
        preds  = outs.argmax(1)
        counts = torch.bincount(preds, minlength=K)
        dom    = counts.max().item() / preds.shape[0]
        lam    = min(LAMBDA_MAX, lam * 1.1) if dom > TAU else max(LAMBDA_MIN, lam * 0.95)
        p      = torch.softmax(outs, dim=1).clamp(min=1e-8).cpu()
        hyx_sum += -(p * p.log()).sum(dim=1).sum().item()
        p_sum   += p.sum(dim=0)
        n_total += len(labels)
        all_preds.append(preds.cpu())
        all_labels.append(labels)

    if (batch_idx + 1) % 10 == 0:
        elapsed = round(time.time() - t_start)
        print(f'  batch {batch_idx+1:3d}/{len(loader)}  dom={dom*100:.1f}%  lam={lam:.3f}  elapsed={elapsed}s')

# ── METRICS ──────────────────────────────────────────────────────────────
preds_all  = torch.cat(all_preds)
labels_all = torch.cat(all_labels)
acc        = preds_all.eq(labels_all).float().mean().item() * 100.0
counts_f   = torch.bincount(preds_all, minlength=K)
dom_final  = counts_f.max().item() / preds_all.shape[0]
active     = (counts_f > 0).sum().item()
collapsed  = int(
    (dom_final > COLLAPSE_DOM_THRESH and active < COLLAPSE_ACTIVE_THRESH)
    or active < COLLAPSE_HARD_THRESH
)
hyx_avg = hyx_sum / n_total
q       = (p_sum / n_total).clamp(min=1e-8)
hy      = -(q * q.log()).sum().item()
mi      = max(hy - hyx_avg, 0.0)
elapsed = round(time.time() - t_start, 1)

print()
print('=' * 60)
print('FINAL RESULT')
print('=' * 60)
print(f'  BMIA-Adaptive  lr={LR}  acc={acc:.1f}%  col={collapsed}  MI={mi:.3f}  [{elapsed}s]')
print('=' * 60)